In [1]:
import pandas as pd
import numpy as np
import re
import gc
from tqdm.auto import tqdm
from soynlp.normalizer import repeat_normalize

# 데이터 불러오기 (구조 오류 및 인코딩 방어)
try:
    df = pd.read_csv('popgame.csv', on_bad_lines='skip', encoding='utf-8-sig')
except:
    df = pd.read_csv('popgame.csv', on_bad_lines='skip', encoding='cp949')

print(f"전체 데이터 수: {len(df)}")
df.head()

전체 데이터 수: 23987


,게임명,장르,추천여부,작성 시점 플레이타임 (분),리뷰 원문,분석된 속성
0,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,982,이전은 선행해 결정된 이벤트라 넘어갔더니 카2나를 재출시하네 퉤ㅗㅗ,NaN
1,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,63158,핵이 진짜 너무 역겨울정도로 많음 경쟁가면 기본적으로 ESP는 있는거같음 제발 쓸데...,NaN
2,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,64740,썅놈들아 최적화 패치좀 해라,NaN
3,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",비추천,12553,실행이 잘 안됨..,NaN
4,PUBG: BATTLEGROUNDS,"액션, 어드벤처, 대규모 멀티플레이어, 무료 플레이",추천,15855,핵 좀 잡바라 시ba,NaN


In [ ]:
Step 1. 노이즈 제거 (Cleaning)

In [2]:
def clean_text(text):
    if not isinstance(text, str): return ""
    
    # 1. 길이 제한 (분석기 성능 저하 방지)
    text = text[:1000] 
    # 2. HTML 및 특수기호 제거
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣ ]', ' ', text)
    # 3. 반복 문자 정규화 (ㅋㅋㅋ -> ㅋㅋ)
    text = repeat_normalize(text, num_repeats=2)
    # 4. 다중 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_review'] = df['리뷰 원문'].apply(clean_text)
print("노이즈 제거 완료")

노이즈 제거 완료


In [ ]:
Step 2, 3, 4. 형태소 분석, 사용자 사전 및 불용어 제거

In [3]:
import konlpy
import jpype
from ckonlpy.tag import Twitter

# JVM 초기화 (이미 시작된 경우 건너뜀)
if not jpype.isJVMStarted():
    konlpy.jvm.init_jvm(max_heap_size=4096)

twitter = Twitter()

# 4. 장르별 사용자 사전 등록
custom_words = ['타격감', '최적화', '로그라이크', '혜자', '망겜', '갓겜', '가성비', '오픈월드', '멀티플레이']
for word in custom_words:
    twitter.add_dictionary(word, 'Noun')

# 3. 불용어 정의
stopwords = ['그리고', '하지만', '진짜', '매우', '너무', '정말', '것', '이', '그', '저', '때', '수', '등', '를', '에', '의', '가', '이런', '저런']

def tokenize(text):
    # 명사 위주 추출
    tokens = twitter.nouns(text)
    # 불용어 제거 및 한 글자 단어 제외
    tokens = [word for word in tokens if word not in stopwords and len(word) > 1]
    return tokens

# 2. 배치 처리를 통한 안전한 토큰화 (멈춤 현상 방지)
def safe_tokenize_batch(df, chunk_size=500):
    all_tokens = []
    for i in tqdm(range(0, len(df), chunk_size), desc="Tokenizing"):
        chunk = df['cleaned_review'].iloc[i:i+chunk_size]
        all_tokens.extend(chunk.apply(tokenize).tolist())
        if i % 1000 == 0: gc.collect() # 메모리 관리
    return all_tokens

df['tokens'] = safe_tokenize_batch(df)
print("토큰화 완료")

c:\Users\clair\anaconda3\Lib\site-packages\konlpy\tag\_okt.py:17: UserWarning: "Twitter" has changed to "Okt" since KoNLPy v0.4.5.
  warn('"Twitter" has changed to "Okt" since KoNLPy v0.4.5.')


Tokenizing:   0%|          | 0/48 [00:00<?, ?it/s]

토큰화 완료


In [ ]:
Step 5. 정수 인코딩 및 패딩 (Vectorization)

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 5-1. 정수 인코딩
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['tokens'])

# 단어 집합 확인 (상위 5개)
print(f"단어 사전 크기: {len(tokenizer.word_index)}")

# 텍스트 -> 정수 시퀀스
sequences = tokenizer.texts_to_sequences(df['tokens'])

# 5-2. 패딩 (Padding)
# 리뷰의 평균 길이 등을 고려하여 최대 길이 설정
max_len = 50 
X_data = pad_sequences(sequences, maxlen=max_len)

print(f"패딩 완료 데이터 셰이프: {X_data.shape}")

단어 사전 크기: 20075
패딩 완료 데이터 셰이프: (23987, 50)


In [5]:
df.to_csv('final_reviewdata.csv', index=False, encoding='utf-8-sig')
print("전처리 완료 및 파일 저장 성공")

전처리 완료 및 파일 저장 성공
